# Embeddings and clusters (exploration, not part of the pipeline)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CoralsINPA/RedSeaCoralIdentifier/blob/claude/awesome-archimedes-kvh4c7/notebooks/explore/embeddings.ipynb)

**This notebook explores; it is not a step of the pipeline.** The pipeline (segment, filter, zero-shot baseline, expert review, linear probe) is notebooks `01` to `04` and never needs this one or its output. See `docs/exploration_embeddings.md`.

Run the first cell, then the others in order. **A GPU is not needed to explore** the existing BioCLIP features (the default). Switch to a GPU runtime (Runtime > Change runtime type > T4) only to run the DINO job.

**What it does.** (a) Optionally embeds every kept crop with DINOv2 (`RUN_EMBED = True`, a GPU job). (b) Explores a features file: PCA, t-SNE and UMAP maps colored by the baseline's predicted genus, K-Means clusters and a density-based (HDBSCAN) clustering with what each cluster is made of, and a PDF of sample crops of every cluster. It is the exploration of the legacy `CoralMasking.ipynb` (cells 12-15 and 19-22), run on the saved manifest and baseline instead of on every mask.

**Which features.** Set `FEATURES` in the config cell. Both files hold one row per kept crop, keyed by `image_name#coral_id`.

- `finetuned-bioclip__masked`: the baseline's 512-d BioCLIP embeddings of the masked crop (written by `02_zero_shot.ipynb`). The default; no job needed.
- `dinov2-s14__masked_square` (or `dinovdeau-l__masked_square`): DINO features of the masked square crop, written by step (a).

**The genus on the maps is the baseline's zero-shot prediction, not a label.** A cluster that is mostly one predicted genus shows that the features and the model agree, not that the genus is right. A cluster of mixed genera shows where the model has no clear preference.

**Safe to run and re-run.** It reads the shared Drive and only creates new files, never overwriting one: the DINO features if they exist are loaded, and the results go into a new folder `<run>/reports/embeddings_<features>_<date>/`.

| Step | What it does | Time (estimates, not yet measured) | Reads | Writes |
|---|---|---|---|---|
| Setup | mounts Drive, updates the clone, installs `coralid[dino,cluster,viz]` | 3-5 min | GitHub | nothing on Drive |
| Config | loads the run config, checks the inputs | seconds | the run folder | nothing |
| (a) DINO job | only with `RUN_EMBED = True`: DINO features of the masked square crop of every kept crop | about 25 min on a T4 (the same photo reading as the baseline), **resumable** | photos, mask JSONs, the manifest, the model weights | `<run>/features/<model>__masked_square.npz` (+ `.meta.json`); progress in `features/parts/` |
| Load | the features joined to the baseline's genus and the manifest's orientation | seconds | the features, the baseline, the manifest | nothing |
| Maps | PCA-50, then t-SNE and UMAP of `N_MAP` random crops, colored by predicted genus | 2-5 min on a CPU (UMAP compiles on its first call) | nothing more | nothing |
| K-Means | `N_CLUSTERS` clusters of all crops, their quality scores, and the genus mix of each | 1-3 min | nothing more | nothing |
| HDBSCAN | density-based clusters (PCA, UMAP, HDBSCAN with the legacy settings) | 2-5 min | nothing more | nothing |
| Save | the tables, figures and a summary | seconds | nothing more | `<run>/reports/embeddings_<features>_<date>/` (a new folder) |
| Gallery | a PDF with `GALLERY_PER_CLUSTER` random crops of every K-Means cluster | 3-8 min (it reads about 300 photos from Drive) | photos, mask JSONs | the same folder |

**What to paste back to the next session:** the printed scores and cluster tables, and a description (or the saved PNGs) of the maps.

In [2]:
import os
import subprocess
import sys

# ================================ SETTINGS: the only place to edit ================================
DATA_ROOT = "/content/drive/MyDrive/colab notebooks/CoralsMasking"  # the Drive folder CoralsMasking
BRANCH = "claude/awesome-archimedes-kvh4c7"
REPO_URL = "https://github.com/CoralsINPA/RedSeaCoralIdentifier"
REPO_DIR = "/content/RedSeaCoralIdentifier"
EXTRAS = "dino,cluster,viz"
# ==================================================================================================

# The package cannot install itself before it exists. A clone left by an earlier session of this
# runtime is brought to the branch head BEFORE anything is imported from it.
if os.path.isdir(f"{REPO_DIR}/.git"):
    steps = [
        ["git", "-C", REPO_DIR, "fetch", "origin", BRANCH],
        ["git", "-C", REPO_DIR, "checkout", BRANCH],
        ["git", "-C", REPO_DIR, "pull", "--ff-only", "origin", BRANCH],
        ["git", "-C", REPO_DIR, "submodule", "update", "--init"],
    ]
else:
    steps = [["git", "clone", "--recurse-submodules", "--branch", BRANCH, REPO_URL, REPO_DIR]]
for step in steps:
    print("$", " ".join(step))
    subprocess.run(step, check=True)
sys.path.insert(0, f"{REPO_DIR}/src")  # enough to import coralid.colab (standard library only)

from coralid.colab import setup  # noqa: E402

env = setup(
    EXTRAS, data_root=DATA_ROOT, branch=BRANCH, repo_url=REPO_URL, repo_dir=REPO_DIR, update=False
)

$ git -C /content/RedSeaCoralIdentifier fetch origin claude/awesome-archimedes-kvh4c7
$ git -C /content/RedSeaCoralIdentifier checkout claude/awesome-archimedes-kvh4c7
$ git -C /content/RedSeaCoralIdentifier pull --ff-only origin claude/awesome-archimedes-kvh4c7
$ git -C /content/RedSeaCoralIdentifier submodule update --init
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
$ /usr/bin/python3 -m pip install -q -e /content/RedSeaCoralIdentifier[dino,cluster,viz]

Installed versions (pip metadata):
  opencv-python            5.0.0.93
  opencv-python-headless   4.14.0.94
  torch                    2.11.0+cpu
  open-clip-torch          None
  matplotlib               3.10.0
  they share one cv2 folder.

coralid 0.1.0 at commit 49b75e4 (branch claude/awesome-archimedes-kvh4c7)

Setup done. No restart needed.


In [5]:
import logging

import torch

from coralid.config import load_config

# ================================ SETTINGS ================================
#FEATURES = "finetuned-bioclip__masked"  # the features to explore: a file stem in <run>/features/
# Options: "dinov2-s14__masked_square" or "dinovdeau-l__masked_square")
FEATURES = "dinovdeau-l__masked_square"
RUN_EMBED = True  # True: run the DINO job first (a GPU runtime, about 25 min, resumable)
EMBED_MODEL = "dinovdeau-l"  # or "dinovdeau-l" | "dinov2-s14"
N_MAP = 6000  # crops in the t-SNE and UMAP maps (a random subset; the maps take minutes)
N_CLUSTERS = 20  # K-Means clusters (the legacy notebook used 50)
RUN_HDBSCAN = True  # the density-based clustering
HDBSCAN_MIN_CLUSTER, HDBSCAN_MIN_SAMPLES = 500, 50  # the legacy settings (CM cell 20)
MAKE_GALLERY = True  # a PDF with sample crops of every K-Means cluster
GALLERY_PER_CLUSTER = 6
# ==========================================================================

logging.basicConfig(level=logging.WARNING, format="%(levelname)s %(name)s: %(message)s")
logging.getLogger("coralid").setLevel(logging.INFO)

cfg = load_config(f"{REPO_DIR}/configs/iou60_sta55_pts32.yaml")  # reads CORALID_DATA_ROOT
for label, path in {
    "manifest": cfg.manifest_path,
    "features": cfg.features_dir / f"{FEATURES}.npz",
    "baseline": cfg.predictions_dir / "finetuned-bioclip__taxonomy_v1__hard20__all.parquet",
}.items():
    print(f"  {label:9} {'ok' if path.exists() else 'not there yet':13} {path}")
print(
    "GPU:",
    torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (fine for exploring)",
)

  manifest  ok            /content/drive/MyDrive/colab notebooks/CoralsMasking/results_all/iou60_sta55_pts32/manifest/manifest.parquet
  features  not there yet /content/drive/MyDrive/colab notebooks/CoralsMasking/results_all/iou60_sta55_pts32/features/dinovdeau-l__masked_square.npz
  baseline  ok            /content/drive/MyDrive/colab notebooks/CoralsMasking/results_all/iou60_sta55_pts32/predictions/finetuned-bioclip__taxonomy_v1__hard20__all.parquet
GPU: none (fine for exploring)


## (a) DINO embeddings (optional, GPU)

`run_embeddings` cuts the masked square crop of every kept mask (the crop the legacy notebook gave DINO), encodes it and saves `features/<model>__masked_square.npz` with its `.meta.json`. It resumes from `features/parts/`, retries a failed photo on the next run, writes the file only when every photo is done, and loads the file instead if it exists. Leave `RUN_EMBED = False` to explore the BioCLIP features.

In [ ]:
if RUN_EMBED:
    from coralid.explore.embed import run_embeddings

    embedded = run_embeddings(cfg, EMBED_MODEL, progress=True)
    FEATURES = embedded.paths.features.stem
    print(
        f"{embedded.status}: {len(embedded.uids)} crops x {embedded.features.shape[1]} "
        f"dimensions\n{embedded.paths.features}"
    )
else:
    print(f"The DINO job is skipped (RUN_EMBED = False). Exploring {FEATURES}.")

INFO:coralid.explore.dino:Loading DINOv2 ViT-S/14 from torch.hub


Downloading: "https://github.com/facebookresearch/dinov2/zipball/main" to /root/.cache/torch/hub/main.zip


/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")


Downloading: "https://dl.fbaipublicfiles.com/dinov2/dinov2_vits14/dinov2_vits14_pretrain.pth" to /root/.cache/torch/hub/checkpoints/dinov2_vits14_pretrain.pth


100%|██████████| 84.2M/84.2M [00:00<00:00, 211MB/s]


Embedding with dinov2-s14:   0%|          | 0/107 [00:00<?, ?it/s]

INFO:coralid.batch:Run 'dinov2-s14__masked_square': wrote part 0, 25 image(s) done


## Load the features

Each crop is described by the baseline's predicted genus and confidence and by the manifest's photo orientation, area and sharpness, in the order of the feature rows.

In [ ]:
from coralid.explore.analysis import load_exploration

data = load_exploration(cfg, FEATURES)
table = data.table
print(
    f"{FEATURES}: {len(table)} crops x {data.X.shape[1]} dimensions "
    f"on {table['image_name'].nunique()} photos"
)
for note in data.notes:
    print("note:", note)
display(table["predicted_genus"].value_counts().rename("crops").to_frame())

In [ ]:
import matplotlib.pyplot as plt

from coralid.explore.clustering import pca_reduce, subsample, tsne_embed, umap_embed
from coralid.explore.plots import category_palette, embedding_scatter

pca = pca_reduce(data.X, 50)
print(f"PCA-50 keeps {pca.total_variance:.1%} of the variance")
at = subsample(len(table), N_MAP)
tsne_xy, umap_xy = tsne_embed(pca.Z[at]), umap_embed(pca.Z[at])
sub = table.iloc[at].assign(
    tsne_x=tsne_xy[:, 0], tsne_y=tsne_xy[:, 1], umap_x=umap_xy[:, 0], umap_y=umap_xy[:, 1]
)
palette = category_palette(table["predicted_genus"])
maps_fig, axes = plt.subplots(1, 2, figsize=(18, 7))
embedding_scatter(
    sub, "tsne_x", "tsne_y", "predicted_genus", axes[0], "t-SNE", s=6, palette=palette, legend=False
)
embedding_scatter(sub, "umap_x", "umap_y", "predicted_genus", axes[1], "UMAP", s=6, palette=palette)
maps_fig.suptitle(f"{FEATURES}: {len(sub)} random crops, colored by the baseline's predicted genus")
plt.show()

In [ ]:
from coralid.explore.analysis import cluster_genus_table, genus_cluster_table
from coralid.explore.clustering import analyze_kmeans

kmeans = analyze_kmeans(data.X, N_CLUSTERS, silhouette_sample=5000)
table = table.assign(cluster=kmeans.labels)
print(
    f"K-Means, {N_CLUSTERS} clusters: cosine silhouette {kmeans.silhouette:.3f} (closer to 1 is "
    f"better), Davies-Bouldin {kmeans.davies_bouldin:.2f} (closer to 0 is better), "
    f"mean compactness {kmeans.compactness.mean():.3f}"
)
clusters = cluster_genus_table(table, "cluster")
genera = genus_cluster_table(table, "cluster")
print("\nWhat each cluster is made of (top_share 1.0 means one predicted genus only):")
display(clusters.round(3))
print("Where each predicted genus goes:")
display(genera.round(3))
sub = sub.assign(cluster=kmeans.labels[at])
cluster_fig, axes = plt.subplots(1, 2, figsize=(18, 7))
embedding_scatter(sub, "umap_x", "umap_y", "cluster", axes[0], "UMAP, K-Means clusters", s=6)
embedding_scatter(
    sub, "umap_x", "umap_y", "confidence", axes[1], "UMAP, baseline confidence", s=6, cmap="viridis"
)
plt.show()

In [ ]:
from coralid.explore.clustering import hdbscan_clusters

hdb, density = None, None
if RUN_HDBSCAN:
    hdb = hdbscan_clusters(
        data.X, min_cluster_size=HDBSCAN_MIN_CLUSTER, min_samples=HDBSCAN_MIN_SAMPLES
    )
    table = table.assign(density_cluster=hdb.labels)
    print(
        f"HDBSCAN: {hdb.n_clusters} clusters, {hdb.n_noise} noise crops "
        f"({hdb.n_noise / len(table):.1%}); silhouette {hdb.silhouette}, "
        f"Davies-Bouldin {hdb.davies_bouldin} (noise left out); cluster -1 is noise"
    )
    density = cluster_genus_table(table, "density_cluster")
    display(density.round(3))
    shown = table.assign(umap1=hdb.umap_features[:, 0], umap2=hdb.umap_features[:, 1]).iloc[at]
    density_fig = embedding_scatter(
        shown.assign(label=shown["density_cluster"].astype(str)),
        "umap1",
        "umap2",
        "label",
        title="HDBSCAN clusters (first two UMAP dimensions)",
        s=6,
    )
    display(density_fig)
else:
    print("HDBSCAN is skipped (RUN_HDBSCAN = False).")

In [ ]:
from coralid.explore.analysis import explore_summary, write_explore_report

tables = {"crops": table, "map_subset": sub, "kmeans_clusters": clusters, "kmeans_genera": genera}
figures = {"maps": maps_fig, "kmeans_maps": cluster_fig}
if density is not None:
    tables["hdbscan_clusters"] = density
    figures["hdbscan_map"] = density_fig
summary = explore_summary(data, pca, kmeans, hdb, n_map=len(sub))
folder = write_explore_report(cfg.reports_dir, FEATURES, tables, figures, summary)
print("Saved (a new folder, nothing was overwritten):", folder)
for file in sorted(folder.iterdir()):
    print("  ", file.name)

In [ ]:
if MAKE_GALLERY:
    from coralid.explore.clustering import sample_per_group
    from coralid.masks import AnnotationStore
    from coralid.viz import crop_gallery_pdf

    view = "square" if FEATURES.endswith("masked_square") else "masked"  # the crop the model saw
    chosen = sample_per_group(
        table.assign(caption=table["uid"] + " " + table["predicted_genus"]),
        "cluster",
        GALLERY_PER_CLUSTER,
    )
    pdf = crop_gallery_pdf(
        chosen,
        folder / "kmeans_gallery.pdf",
        AnnotationStore(cfg.json_dir),
        cfg.images_dir,
        group_col="cluster",
        caption_col="caption",
        crops=cfg.crops,
        view=view,
        title=f"{FEATURES}: random crops of each K-Means cluster",
    )
    print("Saved:", pdf)
else:
    print("The gallery is skipped (MAKE_GALLERY = False).")

## Next

- Paste the printed scores and cluster tables into the next session.
- To compare with DINO features, set `RUN_EMBED = True` on a GPU runtime, run the notebook again, and read the new report folder next to the BioCLIP one.
- `04_supervised_probe.ipynb` (the pipeline's last step) trains a linear probe on the baseline's feature file once labels exist.